In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import data
from check_submission import check_all

ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]
# Walk-forward validation: each of these months is predicted by models trained only
# on the sales before that month, the same way the real test works.
VALIDATION_MONTHS = pd.period_range("2025-12", "2026-05", freq="M")


In [2]:
def validate(
    modelspecs,
    make_model_func=None,
    mode="expanding",
    train_cutoff=None,
    horizon_months=4,
):
    from models.model import make_model

    if make_model_func is None:
        make_model_func = make_model

    listings = data.load("listings")
    results = []
    model_list = []

    if mode == "expanding":
        # Original validation: retrain for every evaluation month
        months = VALIDATION_MONTHS

    elif mode == "fixed_origin":
        # Train once and freeze the model for the entire horizon
        cutoff = pd.Timestamp(train_cutoff)

        if cutoff != cutoff.to_period("M").start_time:
            raise ValueError("train_cutoff must be the first day of a month.")

        if horizon_months < 1:
            raise ValueError("horizon_months must be positive.")

        months = pd.period_range(
            start=cutoff.to_period("M"),
            periods=horizon_months,
            freq="M",
        )

        train = listings.loc[
            listings["date_sold"] < cutoff
        ]

        if train.empty:
            raise ValueError("No training sales before train_cutoff.")

        # Fit each track exactly ONCE
        fixed_models = [
            make_model_func(track, modelspecs=modelspecs).fit(train)
            for track in TRACKS
        ]

        model_list.append(fixed_models)

        print(
            f"Fixed origin: trained on {len(train):,} sales "
            f"before {cutoff.date()}"
        )

    else:
        raise ValueError("mode must be 'expanding' or 'fixed_origin'.")

    for month in months:
        homes = listings.loc[
            listings["date_sold"].dt.to_period("M") == month
        ]

        if mode == "expanding":
            train = listings.loc[
                listings["date_sold"] < month.start_time
            ]

            models = [
                make_model_func(track, modelspecs=modelspecs).fit(train)
                for track in TRACKS
            ]

            model_list.append(models)
        else:
            # Reuse the SAME fitted models for every month
            models = fixed_models

        for track, model in zip(TRACKS, models):
            predicted = model.predict(
                data.hide_answers(homes, track)
            )

            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"],
                "track": track,
                "month": str(month),
                "house_type_name": homes["house_type_name"],
                "sold_price": homes["sold_price"],
                "predicted_price": np.round(predicted),
            }))

        if mode == "expanding":
            print(
                f"{month}: trained on {len(train):,} earlier sales, "
                f"predicted {len(homes):,} homes"
            )
        else:
            print(
                f"{month}: reused frozen models, "
                f"predicted {len(homes):,} homes"
            )

    return pd.concat(results, ignore_index=True), model_list

def bootstrap_ci(values, statistic, repeats=1000):
    """95% interval: recompute the statistic on 1,000 resamples of the homes
    (drawn with replacement) and keep the middle 95% of the results."""
    rng = np.random.default_rng(0)
    results = [statistic(rng.choice(values, size=len(values))) for _ in range(repeats)]
    return [round(float(x), 2) for x in np.percentile(results, [2.5, 97.5])]

def metrics(predictions):
    """Scores for one track. Errors are in percent of the sold price."""
    signed = 100 * (predictions["predicted_price"] - predictions["sold_price"]) / predictions["sold_price"]
    absolute = signed.abs().to_numpy()
    return {
        "n": len(predictions),
        "mape": round(float(np.mean(absolute)), 2),          # mean absolute % error
        "mape_95ci": bootstrap_ci(absolute, np.mean),
        "mdape": round(float(np.median(absolute)), 2),       # median absolute % error
        "mdape_95ci": bootstrap_ci(absolute, np.median),
        "median_signed_error": round(float(np.median(signed)), 2),  # above 0: predicts too high
    }
def report(df):
    # Accept results from earlier notebook runs without refitting the models.
    predictions = pd.concat(df, ignore_index=True) if isinstance(df, list) else df
    scores = {track: metrics(predictions[predictions["track"] == track]) for track in TRACKS}

    (ROOT / "reports").mkdir(exist_ok=True)
    (ROOT / "reports" / "validation_report.json").write_text(json.dumps(scores, indent=2))
    lines = [
        "# Validation report", "",
        f"Walk-forward validation, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}: each month was "
        "predicted by models trained only on the sales before it.",
        "Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.", "",
        "| Track | Homes | MAPE | MdAPE | Median signed error |", "|---|---|---|---|---|"]
    for track, s in scores.items():
        lines.append(f"| {track} | {s['n']:,} | {s['mape']:.2f}% ({s['mape_95ci'][0]:.2f} to {s['mape_95ci'][1]:.2f}) "
                     f"| {s['mdape']:.2f}% ({s['mdape_95ci'][0]:.2f} to {s['mdape_95ci'][1]:.2f}) "
                     f"| {s['median_signed_error']:+.2f}% |")
    lines += [
        "", "MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.",
        "A median signed error above 0 means the model tends to predict too high.",
        "The intervals treat homes as independent. Homes sold in the same month share the same",
        "market swings, so the real uncertainty is somewhat wider.", "",
        "![Error by price range](../figures/error_by_price_range.png)", ""]
    (ROOT / "reports" / "validation_report.md").write_text("\n".join(lines))
    write_latex_table(scores)
    error_by_price_range(predictions)
    print("\n".join(lines))
    print("Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex "
          "and figures/error_by_price_range.png")


def write_latex_table(scores):
    """The same scores as a LaTeX table, for report/report.tex to include."""
    rows = []
    for track, s in scores.items():
        name = track.replace("_", "-")
        rows.append(f"{name} & {s['n']:,} & {s['mdape']:.2f} ({s['mdape_95ci'][0]:.2f}--{s['mdape_95ci'][1]:.2f}) "
                    f"& {s['mape']:.2f} ({s['mape_95ci'][0]:.2f}--{s['mape_95ci'][1]:.2f}) "
                    f"& {s['median_signed_error']:+.2f} \\\\")
    table = "\n".join([
        "% Written by `python run.py report`. Do not edit by hand: rerun the command instead.",
        "\\begin{tabular}{lrccc}", "\\hline",
        "Track & Homes & MdAPE \\% (95\\% CI) & MAPE \\% (95\\% CI) & Median signed error \\% \\\\", "\\hline",
        *rows, "\\hline", "\\end{tabular}"])
    (ROOT / "reports" / "results_table.tex").write_text(table + "\n")


def error_by_price_range(predictions):
    """Figure: how big the errors are, from the cheapest tenth of homes to the dearest.

    Want more figures? Copy this function and group the errors by something else:
    house_type_name, month, or community (add it to the predictions file in validate()).
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colours = {"off_market": "#2a78d6", "list_aware": "#eb6834"}
    for track in TRACKS:
        rows = predictions[predictions["track"] == track]
        error = 100 * (rows["predicted_price"] - rows["sold_price"]).abs() / rows["sold_price"]
        decile = pd.qcut(rows["sold_price"], 10, labels=False) + 1   # 1 = cheapest tenth
        by_decile = error.groupby(decile)
        middle = by_decile.median()
        ax.plot(middle.index, middle.values, marker="o", linewidth=2, color=colours[track], label=track)
        ax.fill_between(middle.index, by_decile.quantile(0.25), by_decile.quantile(0.75),
                        color=colours[track], alpha=0.15, linewidth=0)
    prices = rows["sold_price"].groupby(decile).median()
    ax.set_xticks(prices.index, [f"{d}\n${p / 1e6:.2f}M" for d, p in prices.items()])
    ax.set_xlabel("Sold-price decile (1 = cheapest tenth of homes), with its median price")
    ax.set_ylabel("Absolute % error\n(line: median, band: middle 50%)")
    ax.set_title(f"Validation error by price range, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    (ROOT / "figures").mkdir(exist_ok=True)
    fig.savefig(ROOT / "figures" / "error_by_price_range.png", dpi=150)
    plt.close(fig)

In [3]:
from models.boost_regressor import CatBoostRegressor, LightGBMRegressor, XGBoostRegressor
from models.arima import ArimaResidualModel
from models.model import make_model
from models.direct_listing import DirectListingModel
from models.model import OffMarketModel

CATBOOST_PARAMS = {
    "iterations": 900,
    "depth": 6,
    "learning_rate": 0.04,
    "l2_leaf_reg": 5,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}

LIGHTGBM_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "num_leaves": 31,
    "min_child_samples": 40,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "max_bin": 255,
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

XGBOOST_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "min_child_weight": 5,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "grow_policy": "depthwise",
    "device": "cpu",
    "enable_categorical": True,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}
0
LGB_MODEL_SPECS = [
    (LightGBMRegressor, "Xpt_spatiotemporal__selected_binary", {'n_estimators': 1800, 'learning_rate': 0.052995139011091236, 'max_depth': 6, 'num_leaves': 40, 'min_child_samples': 30, 'reg_lambda': 16.19372143031597, 'reg_alpha': 0.0, 'subsample': 0.8349167453037107, 'subsample_freq': 1, 'colsample_bytree': 0.953443325579757, 'max_bin': 255, 'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}),
]
XGB_MODEL_SPECS = [
    (XGBoostRegressor, "Xpt_spatiotemporal__selected_binary", {'n_estimators': 700, 'learning_rate': 0.09522271657253033, 'max_depth': 8, 'min_child_weight': 16.684619386293466, 'reg_lambda': 19.50331236061207, 'reg_alpha': 0.0, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'device': 'cpu', 'enable_categorical': True, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}),
]
CAT_MODEL_SPECS = [
    (CatBoostRegressor, 'Xpt_spatiotemporal', {'iterations': 1400, 'depth': 8, 'learning_rate': 0.07127609366218435, 'l2_leaf_reg': 4.14053058084107, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'task_type': 'CPU', 'thread_count': -1, 'random_strength': 0.580328206016201}),
]
MODEL_SPECS = XGB_MODEL_SPECS
def custom_make_model(track, modelspecs=None):
    specs = MODEL_SPECS if modelspecs is None else modelspecs

    if track == "list_aware":
        return DirectListingModel(
            target="ratio",
            modelspecs=specs,
        )

    return OffMarketModel(
        specs,
        center_target=True,
    )


# Default to already-saved XGBoost model snapshots instead of fitting twice.
RUN_BASELINE_TRAINING = False
if RUN_BASELINE_TRAINING:
    predictions, models = validate(
        modelspecs=MODEL_SPECS,
        make_model_func=custom_make_model,
    )

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

import data
from save_model import load_model, model_info


def recover_validation(
    model_name="XGBoost_text_binary",
    save_predictions=True,
):
    # 1. Restore the fitted models
    models = load_model(model_name)
    info = model_info(model_name)

    months = pd.period_range("2025-12", "2026-05", freq="M")
    tracks = ["off_market", "list_aware"]

    # 2. Check that the saved models match our validation setup
    assert len(models) == len(months), "Unexpected number of saved months"
    assert all(len(pair) == 2 for pair in models), "Expected two tracks per month"

    metadata = info.get("metadata", {})
    if metadata.get("validation_months"):
        assert metadata["validation_months"] == [str(m) for m in months]
    if metadata.get("tracks"):
        assert metadata["tracks"] == tracks

    # 3. Load original validation properties
    listings = data.load("listings")
    sale_months = listings["date_sold"].dt.to_period("M")

    results = []

    # 4. Reconstruct predictions without fitting
    for month, month_models in zip(months, models):
        homes = listings.loc[sale_months == month]

        for track, model in zip(tracks, month_models):
            hidden = data.hide_answers(homes, track)
            predicted = np.round(model.predict(hidden))

            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"].to_numpy(),
                "track": track,
                "month": str(month),
                "house_type_name": homes["house_type_name"].to_numpy(),
                "sold_price": homes["sold_price"].to_numpy(),
                "predicted_price": predicted,
            }))

        print(f"{month}: recovered {len(homes):,} homes")

    predictions = pd.concat(results, ignore_index=True)

    # 5. Save recovered predictions
    if save_predictions:
        path = Path("outputs/validation_predictions.csv")
        path.parent.mkdir(parents=True, exist_ok=True)
        predictions.to_csv(path, index=False)
        print(f"\nSaved predictions to {path}")

    return predictions, models


from models.spatiotemporal_features import ComparableSalesFeatures

# Restore original behaviour for legacy saved models
ComparableSalesFeatures.age_filter_first = False
ComparableSalesFeatures.min_same_type_neighbors = 0

# Restore the same variables used by the main notebook
if not RUN_BASELINE_TRAINING:
    predictions, models = recover_validation()
else:
    print("Using freshly fitted baseline models")

display(predictions.head())

2025-12: recovered 1,238 homes
2026-01: recovered 1,058 homes
2026-02: recovered 1,442 homes
2026-03: recovered 1,816 homes


KeyboardInterrupt: 

In [7]:
report(predictions)

# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 10.38% (10.18 to 10.58) | 7.82% (7.64 to 7.99) | +0.41% |
| list_aware | 10,069 | 5.20% (5.12 to 5.29) | 4.17% (4.09 to 4.26) | +0.51% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png


In [10]:
# Share these results with notebook.ipynb and the existing report command.
output_path = ROOT / "outputs" / "validation_predictions.csv"
output_path.parent.mkdir(exist_ok=True)
predictions.to_csv(output_path, index=False)

## Full-strength spatiotemporal individual-model comparison (off-market only)

This experiment keeps the original 700-tree XGBoost reference from `predictions, models`. No baseline retraining.

**Full-history refits:** `spacetime_splines` (3-month winner), `geo_context`, original `combined` (contains splines), `comp_type_quota` and `comp_both` (historical reference controls). Each has the **identical tuned estimator settings**, median-centered log-price target, same chronological training cutoff, same training records and held-out properties.

**Previously evaluated corrections:** `tensor_gam` and `rbf_ridge` are independently recalibrated using the three preceding months of truly out-of-time XGBoost errors. They correct **original baseline XGBoost**, not spline XGBoost. No combination is evaluated in this full-strength run.

The output contains independent per-variant/month predictions with restart-safe caching, pooled and monthly MAPE/MdAPE, paired bootstrap differences and house-type diagnostics. The new results live at `outputs/spacetime_full_feature_comparison/` and do not replace your original validation outputs or saved models.

By default `Run All` restores the trusted `saved_models/XGBoost_text_binary` snapshot rather than training a second XGBoost baseline. You can set `RUN_BASELINE_TRAINING=True` in cell 2 to retrain from scratch. The save step is opt-in so existing model files are not overwritten.

**Separate combination experiment:** `spacetime_combination_screen.ipynb` uses only 10k samples / 220 trees for spline + geo, spline + quota, spline + market, etc. before promoting any combination to the expensive final validation.

**Caution:** Dec 2025–May 2026 has already been consulted; treat this as confirmation rather than an untouched final test.


In [ ]:
# Full-history: each feature and residual model evaluated individually.
RUN_FULL_FEATURE_COMPARISON = False  # previous full test already completed

if RUN_FULL_FEATURE_COMPARISON:
    from spacetime_full_feature_comparison import compare_full_spacetime_models
    from IPython.display import display

    full_pooled, full_monthly, full_by_type, full_predictions = compare_full_spacetime_models(
        baseline_predictions=predictions,
        fitted_models=models,
        months=VALIDATION_MONTHS,
        feature_variants=(
            "baseline",
            "spacetime_splines",
            "geo_context",
            "combined",
            "comp_type_quota",
            "comp_both",
        ),
        residual_variants=("tensor_gam", "rbf_ridge"),
        mode="expanding",
        calibration_months=3,
        calibration_cap=None,
        residual_strength=0.5,
        bootstrap_repeats=500,
        output_dir="outputs/spacetime_full_feature_comparison",
        resume=True,
    )
    display(full_pooled)
    display(full_monthly)
    display(full_by_type)
    print("Results saved in outputs/spacetime_full_feature_comparison/")


In [ ]:
# Optional: keep the previously saved trusted baseline intact by default.
SAVE_BASELINE_MODELS = False
if SAVE_BASELINE_MODELS:
    from save_model import save_model
    save_model(
        models, "XGBoost_text_binary",
        metadata={
            "features": "Xpt_spatiotemporal__selected_binary",
            "tracks": list(TRACKS),
            "validation_months": [str(m) for m in VALIDATION_MONTHS],
            "mode": "expanding",
        },
    )


## Full-history spline + model-specific Tensor GAM experiment

The smaller combination screen favored **XGBoost + spatiotemporal splines + Tensor GAM**, with **XGBoost + splines + geographic context + Tensor GAM** also promising. This separate comparison evaluates both on **all 10,069 off-market homes across the original six months**, with the full training history and the exact tuned 700-tree XGBoost parameters.

**Arms:** original XGBoost baseline (reused), splines (base), splines + geographic context (base), spline Tensor GAM correction, spline + geographic context Tensor GAM correction.

**No residual leakage:** each Tensor GAM calibrates on the **three previous sale months** of *out-of-time errors from its own full-history feature model*. The correction does not reuse GAM coefficients or residual labels from the original XGBoost.

The script verifies the manifest/fingerprint from `outputs/spacetime_full_feature_comparison` before reusing previously calculated spline-model predictions; it refits if the cache is not safe or missing. New fits are saved month by month to `outputs/spacetime_full_spline_gam`, which is independent of earlier output folders and resumable.

**Run:** make sure `predictions, models` have been loaded by the notebook's saved-model recovery cell, then execute the following cell. The earlier full comparison is now disabled by default; set its flag to True only if intentionally rerunning it.

**Interpretation:** This is a developmental comparison, not a new untouched holdout. Promote whichever models improve over spline-only *across multiple months*, preferably on both MAPE and MdAPE.


In [ ]:
# Full historical comparison of two feature-specific Tensor GAM corrections.
RUN_FULL_SPLINE_GAM_COMPARISON = True

if RUN_FULL_SPLINE_GAM_COMPARISON:
    from spacetime_full_gam_comparison import compare_full_spline_gam
    from IPython.display import display

    gam_pooled, gam_monthly, gam_by_type, gam_predictions = compare_full_spline_gam(
        baseline_predictions=predictions,
        fitted_models=models,
        months=VALIDATION_MONTHS,
        calibration_months=3,
        calibration_cap=None,
        residual_strength=0.5,  # matches smaller combination screen
        bootstrap_repeats=500,
        previous_feature_dir="outputs/spacetime_full_feature_comparison",
        output_dir="outputs/spacetime_full_spline_gam",
        resume=True,
    )
    display(gam_pooled)
    display(gam_monthly)
    display(gam_by_type)
    print("Saved results in outputs/spacetime_full_spline_gam/")
